# 🔍 競技プログラミング提出物 AI生成コード判定ツール
## Contest Code AI Detector

このノートブックは、プログラミングコンテストに提出された Python コードが **AI生成** か **人間が記述** したものかを判定します。

### 設計方針
- ❌ **偽陽性（人間 → AI 誤判定）は絶対に許容しない**
- ✅ 偽陰性（AI → 人間 誤判定）は許容する
- 📝 競技プログラミング特有の特徴（最小限の変数名・コメント無し・汎用的命名）は AI の根拠にしない

### 使い方
1. セルを上から順に実行してください（`Shift+Enter`）
2. **③ のセル**で `.txt` 形式の Python ファイルをアップロードしてください
3. 判定結果と根拠が表示されます


## ① 分析エンジン（実行してください）

In [ ]:
"""
Core heuristics for the contest code AI detector.
"""

import ast
import re


THRESHOLD = 40

_ALGORITHMIC_COMMENT_PATTERN = re.compile(
    r'#\s+(?:Initialize|Compute|Calculate|Process|Handle|Check\s+if|Find\s+the|'
    r'Sort\s+the|Build\s+the|Create\s+the|Update\s+the|Read\s+the|Parse\s+the|'
    r'Store\s+the|Iterate|Traverse|Use\s+a|Get\s+the|Set\s+the|Add\s+the|'
    r'Remove\s+the|Return\s+the|Time\s+[Cc]omplexity|Space\s+[Cc]omplexity)'
    r'[\w\s,.:]*'
)


def _parse_tree(source_code):
    try:
        return ast.parse(source_code), None
    except SyntaxError as exc:
        return None, str(exc)


def _imported_modules(tree):
    modules = set()
    if tree is None:
        return modules

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                modules.add(alias.name.split('.')[0])
        elif isinstance(node, ast.ImportFrom) and node.module:
            modules.add(node.module.split('.')[0])
    return modules


def _attribute_name(node):
    if isinstance(node, ast.Name):
        return node.id
    if isinstance(node, ast.Attribute):
        base = _attribute_name(node.value)
        if base:
            return f'{base}.{node.attr}'
        return node.attr
    return ''


def _has_dataclass_decorator(tree):
    if tree is None:
        return False

    for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef):
            for decorator in node.decorator_list:
                name = _attribute_name(decorator)
                if name == 'dataclass' or name.endswith('.dataclass'):
                    return True
    return False


def _is_main_guard_compare(node):
    if not isinstance(node, ast.Compare):
        return False
    if len(node.ops) != 1 or not isinstance(node.ops[0], ast.Eq):
        return False
    if len(node.comparators) != 1:
        return False

    left = node.left
    right = node.comparators[0]
    left_value = left.value if isinstance(left, ast.Constant) else None
    right_value = right.value if isinstance(right, ast.Constant) else None

    return (
        (isinstance(left, ast.Name) and left.id == '__name__' and right_value == '__main__')
        or (isinstance(right, ast.Name) and right.id == '__name__' and left_value == '__main__')
    )


def _has_main_guard(tree):
    if tree is None:
        return False
    return any(
        isinstance(node, ast.If) and _is_main_guard_compare(node.test)
        for node in ast.walk(tree)
    )


def _has_zero_arg_main_def(tree):
    if tree is None:
        return False

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == 'main':
            args = node.args
            arg_count = len(args.posonlyargs) + len(args.args) + len(args.kwonlyargs)
            if arg_count == 0 and args.vararg is None and args.kwarg is None:
                return True
    return False


def _has_zero_arg_main_call(tree):
    if tree is None:
        return False

    for node in ast.walk(tree):
        if (
            isinstance(node, ast.Call)
            and isinstance(node.func, ast.Name)
            and node.func.id == 'main'
            and not node.args
            and not node.keywords
        ):
            return True
    return False


def _has_dunder_all_assignment(tree):
    if tree is None:
        return False

    for node in tree.body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id == '__all__':
                    return True
        elif isinstance(node, ast.AnnAssign):
            if isinstance(node.target, ast.Name) and node.target.id == '__all__':
                return True
    return False


def _module_imported_by_regex(source_code, module_name):
    pattern = (
        rf'^\s*(?:from\s+{re.escape(module_name)}(?:\.|\s+import)'
        rf'|import\s+{re.escape(module_name)}\b)'
    )
    return bool(re.search(pattern, source_code, re.MULTILINE))


def analyze_contest_code(source_code: str) -> dict:
    """
    競技プログラミング提出物が AI 生成か人間記述かを判定する。

    スコアが THRESHOLD 以上 → AI 生成と判定
    スコアが THRESHOLD 未満 → 人間記述と判定
    """

    signals = []
    total_score = 0
    lines = source_code.splitlines()

    tree, parse_error = _parse_tree(source_code)
    modules = _imported_modules(tree)

    has_main_guard = _has_main_guard(tree) or (
        tree is None
        and bool(re.search(r'if\s+__name__\s*==\s*["\']__main__["\']', source_code))
    )
    if has_main_guard:
        score = 60
        total_score += score
        signals.append({
            'level': '強',
            'score': score,
            'title': '`if __name__ == "__main__":` ガード',
            'detail': (
                'スクリプト末尾の `if __name__ == "__main__":` ブロックが検出されました。'
                '競技プログラミングの提出物は単一スクリプトとして直接実行されるため、'
                'このガードは不要であり、AI 生成コードや本番向けモジュールに典型的なパターンです。'
            ),
        })

    if tree is not None:
        docstring_locs = []
        if ast.get_docstring(tree):
            docstring_locs.append('モジュールレベル')
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                if ast.get_docstring(node):
                    docstring_locs.append(f'`{node.name}`')
        if docstring_locs:
            score = min(50 + (len(docstring_locs) - 1) * 10, 80)
            total_score += score
            signals.append({
                'level': '強',
                'score': score,
                'title': f'ドキュメント文字列 (docstring) × {len(docstring_locs)} 箇所',
                'detail': (
                    f'{", ".join(docstring_locs[:4])} にて三重引用符による docstring が検出されました。'
                    '競技プログラミングの提出物では関数・クラスに説明文字列を付けることは極めて稀です。'
                ),
            })

    has_typing = ('typing' in modules) or (
        tree is None and _module_imported_by_regex(source_code, 'typing')
    )
    if has_typing:
        score = 50
        total_score += score
        signals.append({
            'level': '強',
            'score': score,
            'title': '`typing` モジュールのインポート',
            'detail': (
                '型ヒント管理のための `typing` モジュールがインポートされています。'
                '競技プログラミングでは型注釈の厳密な管理は必要とされず、このインポートは見られません。'
            ),
        })

    has_try_except = (
        any(isinstance(node, ast.Try) for node in ast.walk(tree))
        if tree is not None
        else bool(re.search(r'^\s*try\s*:', source_code, re.MULTILINE))
    )
    if has_try_except:
        score = 40
        total_score += score
        signals.append({
            'level': '強',
            'score': score,
            'title': '`try-except` ブロック',
            'detail': (
                '例外処理ブロックが検出されました。'
                '競技プログラミングではジャッジの入力は問題の制約を満たすことが保証されているため、'
                '例外処理は通常記述しません。'
            ),
        })

    has_logging = ('logging' in modules) or (
        tree is None and _module_imported_by_regex(source_code, 'logging')
    )
    if has_logging:
        score = 55
        total_score += score
        signals.append({
            'level': '強',
            'score': score,
            'title': '`logging` モジュールのインポート',
            'detail': 'ログ出力管理モジュールがインポートされています。競技プログラミングでは使用しません。',
        })

    has_argparse = ('argparse' in modules) or (
        tree is None and _module_imported_by_regex(source_code, 'argparse')
    )
    if has_argparse:
        score = 60
        total_score += score
        signals.append({
            'level': '強',
            'score': score,
            'title': '`argparse` モジュールのインポート',
            'detail': 'コマンドライン引数解析モジュールがインポートされています。競技プログラミングでは使用しません。',
        })

    has_dataclass = (
        'dataclasses' in modules
        or _has_dataclass_decorator(tree)
        or (tree is None and (
            _module_imported_by_regex(source_code, 'dataclasses')
            or bool(re.search(r'@dataclass\b', source_code))
        ))
    )
    if has_dataclass:
        score = 45
        total_score += score
        signals.append({
            'level': '強',
            'score': score,
            'title': '`dataclasses` / `@dataclass` の使用',
            'detail': 'データクラスモジュールまたはデコレータが検出されました。競技プログラミングでは使用しません。',
        })

    has_abc = ('abc' in modules) or (
        tree is None and _module_imported_by_regex(source_code, 'abc')
    )
    if has_abc:
        score = 40
        total_score += score
        signals.append({
            'level': '強',
            'score': score,
            'title': '`abc` (抽象基底クラス) のインポート',
            'detail': '抽象基底クラスモジュールがインポートされています。競技プログラミングでは使用しません。',
        })

    has_main_pattern = (
        (_has_zero_arg_main_def(tree) and _has_zero_arg_main_call(tree))
        if tree is not None
        else bool(re.search(r'^\s*def\s+main\s*\(\s*\)\s*:', source_code, re.MULTILINE))
        and bool(re.search(r'^\s*main\s*\(\s*\)', source_code, re.MULTILINE))
    )
    if has_main_pattern:
        score = 35
        total_score += score
        signals.append({
            'level': '中',
            'score': score,
            'title': '`def main():` + `main()` 呼び出しパターン',
            'detail': (
                '`def main()` 関数の定義と明示的な呼び出しが検出されました。'
                'AI 生成コードに非常に多い構造ですが、競技プログラミングでは不要です。'
            ),
        })

    has_dunder_all = _has_dunder_all_assignment(tree) or (
        tree is None and bool(re.search(r'^__all__\s*=', source_code, re.MULTILINE))
    )
    if has_dunder_all:
        score = 25
        total_score += score
        signals.append({
            'level': '中',
            'score': score,
            'title': '`__all__` の定義',
            'detail': 'モジュール公開 API を定義する `__all__` が検出されました。競技プログラミングでは使用しません。',
        })

    has_pathlib = ('pathlib' in modules) or (
        tree is None and _module_imported_by_regex(source_code, 'pathlib')
    )
    if has_pathlib:
        score = 30
        total_score += score
        signals.append({
            'level': '中',
            'score': score,
            'title': '`pathlib` モジュールのインポート',
            'detail': (
                'パス操作モジュールがインポートされています。'
                '競技プログラミングでは標準入力を使用するためファイルパス操作は不要です。'
            ),
        })

    has_json = ('json' in modules) or (
        tree is None and _module_imported_by_regex(source_code, 'json')
    )
    if has_json:
        score = 25
        total_score += score
        signals.append({
            'level': '中',
            'score': score,
            'title': '`json` モジュールのインポート',
            'detail': (
                'JSON 処理モジュールがインポートされています。'
                '競技プログラミングの入出力は通常テキスト形式であり、JSON は使用しません。'
            ),
        })

    algorithmic_comments = _ALGORITHMIC_COMMENT_PATTERN.findall(source_code)
    if len(algorithmic_comments) >= 3:
        score = min(len(algorithmic_comments) * 6, 24)
        total_score += score
        examples = [comment.strip()[:60] for comment in algorithmic_comments[:2]]
        signals.append({
            'level': '弱',
            'score': score,
            'title': f'英語の説明的アルゴリズムコメント × {len(algorithmic_comments)} 箇所',
            'detail': (
                f'アルゴリズムの手順を英語で説明するコメントが {len(algorithmic_comments)} 箇所検出されました。'
                f'例: {examples}。競技プログラミングの人間提出物では稀なパターンです。'
            ),
        })

    if tree is not None:
        assert_count = sum(1 for node in ast.walk(tree) if isinstance(node, ast.Assert))
        if assert_count >= 2:
            score = 15
            total_score += score
            signals.append({
                'level': '弱',
                'score': score,
                'title': f'`assert` 文 × {assert_count} 個',
                'detail': '複数のデバッグ用 `assert` 文が検出されました。競技プログラミングでは通常使用しません。',
            })

    blank_count = sum(1 for line in lines if not line.strip())
    if len(lines) >= 20:
        blank_ratio = blank_count / len(lines)
        if blank_ratio > 0.38:
            score = 12
            total_score += score
            signals.append({
                'level': '弱',
                'score': score,
                'title': f'高い空行密度 ({blank_ratio:.0%})',
                'detail': (
                    f'全 {len(lines)} 行のうち {blank_count} 行（{blank_ratio:.0%}）が空行です。'
                    'AI は可読性向上のために多くの空行を挿入する傾向があります。'
                ),
            })

    return {
        'is_ai': total_score >= THRESHOLD,
        'score': total_score,
        'threshold': THRESHOLD,
        'signals': signals,
        'parse_error': parse_error,
        'stats': {
            'total_lines': len(lines),
            'blank_lines': blank_count,
            'non_blank_lines': len(lines) - blank_count,
        },
    }


def decode_uploaded_text(content: bytes) -> str:
    for encoding in ['utf-8-sig', 'utf-8', 'shift-jis', 'cp932']:
        try:
            return content.decode(encoding)
        except (UnicodeDecodeError, ValueError):
            pass
    return content.decode('utf-8', errors='replace')


## ② 表示ユーティリティ（実行してください）

In [ ]:
from IPython.display import HTML, display


def display_result(result: dict, filename: str = '') -> None:
    """????? HTML ???????????"""

    if result['is_ai']:
        verdict_ja = 'AI ?????????'
        verdict_en = 'Likely AI-generated'
        main_color = '#c62828'
        main_bg = '#ffebee'
        icon = '&#x1F916;'
    else:
        verdict_ja = '?????????????'
        verdict_en = 'Likely human-written'
        main_color = '#2e7d32'
        main_bg = '#e8f5e9'
        icon = '&#x1F464;'

    level_colors = {'?': '#c62828', '?': '#e65100', '?': '#f57f17'}
    bar_pct = min(result['score'] / max(result['threshold'] * 2, 1) * 100, 100)

    if result['signals']:
        signal_items = ''
        for signal in result['signals']:
            color = level_colors.get(signal['level'], '#555')
            signal_items += (
                f'<div style="margin:8px 0;padding:10px 14px;border-left:4px solid {color};'
                f'background:#fafafa;border-radius:0 6px 6px 0;">'
                f'<div style="margin-bottom:4px;">'
                f'<span style="background:{color};color:#fff;padding:2px 7px;'
                f'border-radius:3px;font-size:12px;font-weight:bold;">'
                f'{signal["level"]}???? &nbsp;+{signal["score"]}</span>'
                f'<strong style="margin-left:8px;font-size:14px;">{signal["title"]}</strong></div>'
                f'<div style="color:#444;font-size:13px;line-height:1.6;">{signal["detail"]}</div>'
                f'</div>'
            )
    else:
        signal_items = (
            '<div style="color:#555;font-style:italic;padding:8px;">'
            'AI ?????????????????????</div>'
        )

    parse_warning = ''
    if result['parse_error']:
        parse_warning = (
            f'<div style="background:#fff3e0;border:1px solid #ffb300;border-radius:6px;'
            f'padding:8px 12px;margin-bottom:12px;font-size:13px;color:#e65100;">'
            f'&#x26A0; ???????? AST ???????????: {result["parse_error"]}'
            f'</div>'
        )

    fname_html = (
        f'<div style="margin-top:6px;font-size:13px;color:#888;">????: {filename}</div>'
        if filename else ''
    )

    html = (
        '<div style="font-family:Helvetica Neue,Arial,sans-serif;max-width:780px;'
        'margin:10px 0;line-height:1.5;">'
        f'<div style="background:{main_bg};border:2px solid {main_color};border-radius:10px;'
        f'padding:18px 22px;margin-bottom:14px;">'
        f'<div style="font-size:26px;font-weight:bold;color:{main_color};">'
        f'{icon}&nbsp; {verdict_ja}</div>'
        f'<div style="color:#666;font-size:14px;margin-top:2px;">{verdict_en}</div>'
        f'{fname_html}</div>'
        f'<div style="background:#f5f5f5;border-radius:8px;padding:12px 16px;margin-bottom:14px;">'
        f'<div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">'
        f'<span><strong>AI ???:</strong> {result["score"]} ?</span>'
        f'<span style="color:#888;font-size:13px;">??????: {result["threshold"]} ?</span></div>'
        f'<div style="background:#e0e0e0;border-radius:4px;height:10px;overflow:hidden;">'
        f'<div style="background:{main_color};width:{bar_pct:.1f}%;height:100%;border-radius:4px;"></div></div>'
        f'<div style="display:flex;gap:20px;margin-top:8px;font-size:13px;color:#555;">'
        f'<span>???: {result["stats"]["total_lines"]}</span>'
        f'<span>????: {result["stats"]["non_blank_lines"]}</span>'
        f'<span>??: {result["stats"]["blank_lines"]}</span></div></div>'
        f'{parse_warning}'
        f'<div style="font-size:15px;font-weight:bold;margin-bottom:8px;">'
        f'????????? ({len(result["signals"])} ?)</div>'
        f'{signal_items}'
        '<div style="margin-top:16px;padding:12px 14px;background:#e3f2fd;'
        'border-radius:8px;font-size:12px;color:#1565c0;line-height:1.6;">'
        '<strong>&#x1F4CC; ??:</strong> '
        '???????????????????????????????<br>'
        '???????????????????????????????????'
        'AI ????????<strong>????????</strong>?<br>'
        '??????&#x2192;AI ????????????????'
        'AI ???????????????????????????????????'
        '</div></div>'
    )
    display(HTML(html))


## ③ ファイルをアップロードして判定

In [ ]:
# ??????????????????????????????????????????????
#  ?????????????????
# ??????????????????????????????????????????????
from google.colab import files as _colab_files

print("Python ???? .txt ?????????????????")
print("??????????????????????")
print()

uploaded = _colab_files.upload()

if not uploaded:
    print("????????????????????")
else:
    for filename, content in uploaded.items():
        source_code = decode_uploaded_text(content)
        print(f"\n{'='*60}\n????: {filename}\n{'='*60}")
        result = analyze_contest_code(source_code)
        display_result(result, filename)

        preview = '\n'.join(source_code.splitlines()[:30])
        if len(source_code.splitlines()) > 30:
            preview += f'\n... (?? {len(source_code.splitlines()) - 30} ???)'
        print("\n--- ????? (??30?) ---")
        print(preview)


## （オプション）④ バッチテスト: ZIP で複数ファイルを一括判定

複数の `.txt` ファイルを ZIP にまとめてアップロードすると、一括判定できます。

In [ ]:
# ZIP ???????? .txt ????????????????????
import io
import zipfile
from google.colab import files as _colab_files

print("ZIP ?????.txt ???????????????????????")
uploaded_zip = _colab_files.upload()

for zip_filename, zip_content in uploaded_zip.items():
    if not zip_filename.endswith('.zip'):
        print(f"  ? {zip_filename} ? ZIP ????????????????")
        continue

    with zipfile.ZipFile(io.BytesIO(zip_content)) as zip_file:
        txt_files = sorted(
            name for name in zip_file.namelist()
            if name.endswith('.txt') and not name.startswith('__')
        )
        print(f"ZIP ?? .txt ?????: {len(txt_files)}\n")
        ai_count = 0
        human_count = 0

        for name in txt_files:
            source_code = decode_uploaded_text(zip_file.read(name))
            result = analyze_contest_code(source_code)
            label = '?? AI  ' if result['is_ai'] else '?? ??'
            signal_summary = ', '.join(signal['title'] for signal in result['signals']) or '??'
            print(
                f"  {label}  {name:<40}  score={result['score']:<4}  "
                f"????: {signal_summary}"
            )
            if result['is_ai']:
                ai_count += 1
            else:
                human_count += 1

        print(f"\n????: ??={human_count} ?  AI={ai_count} ?  ??={len(txt_files)} ?")
